<a href="https://colab.research.google.com/github/ZaxkyyOfficial/Flyrank-AI/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZaxkyyOfficial/Flyrank-AI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
Unit of analysis: A single row in the fact table represents a unique combination of a web page (content item) and a client on a specific day of observation.

Time window: I am using the "mid-panel month"—specifically, data from March 1, 2026, to March 31, 2026. Using a month from the middle of the panel ensures I have both historical data (for features) and subsequent future data (for labels), while keeping the data from the final month (June 2026) reserved for a blind test.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import os
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score

# Setup Token dan Koneksi DuckDB
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

print("Koneksi DuckDB berhasil disiapkan untuk menembak data bulan Maret 2026.")

Koneksi DuckDB berhasil disiapkan untuk menembak data bulan Maret 2026.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features (Max 5):

impressions_prev30: Known at the time of decision, as impressions from the preceding 30 days are recorded before the decision is made.

clicks_prev30: Known at the time of decision, as the click history occurred entirely in the past.

content_age_days: Known at the time of decision, as the publication date is static and the content's age can be calculated instantly.

avg_position: Known at the time of decision, as historical average rankings are stored in Search Console.

word_count: Known at the time of decision, as the article length is definitively known from the current live version.

Label / Proxy: target_is_declining (Value of 1 if impressions drop >20% in the future, 0 otherwise).

Context: client_hash_id and content_hash_id (used as relational keys, not as prediction features).

Excluded: priority_score, health_score, or action flags from the FlyRank product. Reason: I intentionally excluded them to prevent target leakage (where the model "cheats" by using the existing ranking system's future data).

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# MENDEMONSTRASIKAN "THE TRAP" (Kebocoran Data)
print("=== THE TRAP (DATA LEAKAGE LESSON) ===")

# --- PENGAMAN TOKEN: Memastikan DuckDB ingat passwordnya sebelum menarik data ---
import os
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
# ---------------------------------------------------------------------------------

# Tarik sampel cepat untuk eksperimen
df_trap = con.sql(f"""
    SELECT
        gsc_impressions AS impressions_past,
        gsc_clicks AS clicks_past,
        gsc_avg_position AS avg_position,
        CASE WHEN gsc_impressions < 50 THEN 1 ELSE 0 END AS target_is_declining,
        -- INI ADALAH JEBAKAN (Memasukkan label ke dalam fitur secara terselubung)
        CASE WHEN gsc_impressions < 50 THEN 1 ELSE 0 END AS LEAKED_future_trend
    FROM {FACT_MARCH}
    LIMIT 10000
""").df()

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score

# 1. Model dengan bocoran (Mendapat nilai sempurna karena curang)
X_trap = df_trap[['impressions_past', 'clicks_past', 'avg_position', 'LEAKED_future_trend']]
y = df_trap['target_is_declining']
X_tr, X_te, y_tr, y_te = train_test_split(X_trap, y, test_size=0.3, random_state=42)
trap_model = RandomForestClassifier(n_estimators=10, random_state=42).fit(X_tr, y_tr)
print(f"Skor DENGAN Leakage (Palsu): Precision = {precision_score(y_te, trap_model.predict(X_te)):.3f}")

# 2. Model Jujur (Menghapus kolom bocoran)
X_honest = df_trap[['impressions_past', 'clicks_past', 'avg_position']]
X_tr_h, X_te_h, y_tr_h, y_te_h = train_test_split(X_honest, y, test_size=0.3, random_state=42)
honest_model = RandomForestClassifier(n_estimators=10, random_state=42).fit(X_tr_h, y_tr_h)
print(f"Skor JUJUR (Leak dihapus): Precision = {precision_score(y_te_h, honest_model.predict(X_te_h)):.3f}")

=== THE TRAP (DATA LEAKAGE LESSON) ===
Skor DENGAN Leakage (Palsu): Precision = 1.000
Skor JUJUR (Leak dihapus): Precision = 1.000


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I verified three facts from this data slice:

Grain: Confirmed that the combination of client, content, and date is truly unique, with no duplicates.

Row count & date span: Ensured the data covers only March 2026.

Availability: Checked data availability (how many rows actually contain active GA4 data).

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=== VERIFICATION QUERIES ===")

# 1. Prove the grain
grain_check = con.sql(f"""
    SELECT COUNT(*) as total,
           COUNT(DISTINCT client_hash_id || content_hash_id || CAST(report_date AS VARCHAR)) as dist
    FROM {FACT_MARCH}
""").df()
print(f"1. Grain Check: Total baris ({grain_check['total'][0]:,}) == Unik ({grain_check['dist'][0]:,})")

# 2. Row count & date span
date_span = con.sql(f"""
    SELECT COUNT(*) as total, MIN(report_date) as start_d, MAX(report_date) as end_d
    FROM {FACT_MARCH}
""").df()
print(f"2. Date Span: {date_span['total'][0]:,} baris. Window dari {date_span['start_d'][0]} hingga {date_span['end_d'][0]}")

# 3. Availability Check (IS TRUE)
avail = con.sql(f"""
    SELECT COUNT(*) as valid
    FROM {FACT_MARCH}
    WHERE ga4_data_available IS TRUE
""").df()
print(f"3. Availability Check: Terdapat {avail['valid'][0]:,} baris yang memiliki data GA4 valid (IS TRUE).")

=== VERIFICATION QUERIES ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1. Grain Check: Total baris (9,841,378) == Unik (9,841,378)
2. Date Span: 9,841,378 baris. Window dari 2026-03-01 00:00:00 hingga 2026-03-31 00:00:00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

3. Availability Check: Terdapat 413,966 baris yang memiliki data GA4 valid (IS TRUE).


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
This data slice from March 2026 cannot inform me about long-term seasonal effects. A model trained solely on March data might fail to anticipate the natural traffic decline that occurs at the end of the year. Furthermore, this panel has an unbalanced history; some clients have a late `gsc_data_start`, making it appear as though they have no traffic, when in reality, the system simply hadn't recorded it yet.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.